# W04_6 astra — 모델이 스크립트를 쓴다

<a href="https://colab.research.google.com/github/kloud80/dataminig_cau/blob/main/%EC%A4%91%EC%95%99%EB%8C%8026%ED%8A%B9%EA%B0%95/notebooks/w04_colab/W04_6%20astra%20%E2%80%94%20%EB%AA%A8%EB%8D%B8%EC%9D%B4%20%EC%8A%A4%ED%81%AC%EB%A6%BD%ED%8A%B8%EB%A5%BC%20%EC%93%B4%EB%8B%A4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

5번 노트북의 방식(v0.3)은 **스키마가 담을 수 있는 것만** 지을 수 있다. 아이파크타워 외벽의 거대한 원형 조형,
비스듬히 박힌 붉은 상자들, 곡면 저층부 — 스키마에 칸이 없으면 표현 자체가 불가능하다.
대장 수치가 사진과 다를 때도 수치 쪽이 이겨 실물과 멀어진다.

`addr2blend` v0.4 는 반대로 간다. **반드시 맞아야 하는 것**(대지 형상·좌표계·높이)만 고정으로 주고,
색·재질·형태는 사진을 본 모델이 정한다. 그리고 그 결과를 JSON 이 아니라 **파이썬 스크립트**로 받는다.

| | v0.3 BDS (5번 노트북) | v0.4 자유 생성 (이 노트북) |
|---|---|---|
| 모델이 내는 것 | 비율 JSON (스키마 강제) | **빌더 API 7개만 쓰는 파이썬 스크립트** |
| 형상을 만드는 것 | 우리 코드 (결정론적) | 모델이 쓴 코드 → 우리 샌드박스에서 실행 |
| 표현 범위 | 스키마의 칸만큼 | API 로 조합 가능한 모든 것 |
| 고정하는 것 | 치수 전부 (공공데이터) | 대지 외곽선 · 좌표계 · 높이 |
| 위험 | 숫자가 이상하면 조인다 | **남이 쓴 코드를 실행한다** → 검사기 + 샌드박스 |

| 단계 | 이 노트북 | addr2blend 원본 |
|---|---|---|
| 1 | 고정 입력 만들기 (외곽선·높이·대장 요약·지붕 영상·사진) | `stages/s30_freeform.py` |
| 2 | 프롬프트 + API 문서 7개 함수 | `bds/freeform.py` `PROMPT`, `API_DOC` |
| 3 | 모델 호출 → 스크립트 | `freeform.generate` (`gpt-6-astra`) |
| 4 | **정적 검사기** (AST 허용 목록) | `freeform.check_script` |
| 5 | **샌드박스 실행** (빈 `__builtins__` + 함수 7개) | `blender/build_freeform.py` `run_script` |
| 6 | 7개 함수를 trimesh 로 구현 → `.glb` | 원본은 Blender `bmesh` |

> **키가 없어도 끝까지 돈다.** 실제 run 에서 `gpt-6-astra` 가 쓴 488줄 스크립트(`data/ipark/astra_build.py`)를
> **똑같은 검사기와 샌드박스**에 통과시켜 짓는다.

## 1. 준비

In [ ]:
import os, sys, io, json, math, time, base64, subprocess, importlib
from pathlib import Path

# ── 데이터 찾기 ─────────────────────────────────────────────
REPO_URL = "https://github.com/kloud80/dataminig_cau.git"  # 강의 저장소
REPO_DIR = "중앙대26특강/notebooks/w04_colab"   # 저장소 안에서 이 노트북들이 있는 폴더

def find_base():
    for b in [Path.cwd(), Path.cwd() / REPO_DIR, Path("/content") / REPO_DIR, Path("/content/repo") / REPO_DIR]:
        if (b / "data" / "ipark" / "s04_geometry.json").exists():
            return b
    return None

BASE = find_base()
if BASE is None and REPO_URL:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO_URL, "/content/repo"], check=False)
    subprocess.run(["git", "-C", "/content/repo", "sparse-checkout", "set", REPO_DIR], check=False)
    BASE = find_base()
if BASE is None:
    raise SystemExit(
        "data/ipark 폴더를 찾지 못했습니다.\n"
        "  (1) 네트워크를 확인하고 이 셀을 다시 실행 (강의 저장소에서 자동으로 받는다)\n"
        "  (2) 또는 왼쪽 파일 탭에 w04_colab 폴더(data/ipark 포함)를 통째로 올린 뒤 다시 실행")
DATA = BASE / "data" / "ipark"
OUT = BASE / "out"
OUT.mkdir(exist_ok=True)
sys.path.insert(0, str(BASE))          # w04_tools.py (보기용 도우미)

# ── 패키지 (Colab 에 없는 것만 설치) ────────────────────────
def need(mod, pip_name=None):
    try:
        return importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or mod], check=True)
        return importlib.import_module(mod)

import ast, re, copy
from collections import Counter
import numpy as np
from PIL import Image
need("shapely")
need("mapbox_earcut", "mapbox-earcut")
trimesh = need("trimesh")
from shapely.geometry import Polygon, Point
from w04_tools import PartSet, box_vf, render, show_glb, sha, hex_rgb

# ── API 키: Colab 보안 비밀 → 환경변수. 값은 절대 출력하지 않는다 ──
def get_key(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v.strip()
    except Exception:
        pass
    v = os.environ.get(name, "").strip()
    return v or None

OPENAI_KEY = get_key("OPENAI_API_KEY")
print("OPENAI_API_KEY:", "있음" if OPENAI_KEY else "없음 → 실제 run 의 astra 스크립트로 진행")
print("데이터:", DATA)
print("결과물:", OUT)

In [ ]:
# 한글 폰트 — 그래프 글자가 네모로 깨지지 않게 (Colab 은 나눔고딕을 설치, 윈도는 맑은 고딕)
import platform
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
FONT_PATH = None
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgunbd.ttf'
else:
    nanum = '/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf'
    if not os.path.exists(nanum):
        os.system('apt-get -qq install -y fonts-nanum > /dev/null')
    if os.path.exists(nanum):
        fm.fontManager.addfont(nanum)
        fm.fontManager.addfont(nanum.replace('Bold', ''))
        plt.rcParams['font.family'] = 'NanumGothic'
        FONT_PATH = nanum
plt.rcParams['axes.unicode_minus'] = False

## 2. 고정 입력 — 지도에 다시 얹으려면 반드시 맞아야 하는 것

나중에 이 건물을 지도·거리 씬에 다른 건물과 나란히 놓으려면 **좌표계·외곽선·높이**는 어긋나면 안 된다.
이것만 "FIXED" 로 준다. 대장의 층수·용도·준공연도는 **규모를 알려 주는 참고**로만 준다 — 외관을 지시하지 않는다.

지붕 영상은 4번 노트북의 `out/roof_crop.png`(기복변위 보정·주축 정렬)를 쓰고, 없으면 실제 run 의 것을 쓴다.

In [ ]:
prior = json.loads((DATA / "s03_prior.json").read_text(encoding="utf-8"))
geom = json.loads((DATA / "s04_geometry.json").read_text(encoding="utf-8"))
FP = np.array(geom["footprint"])
H = float(geom["height_m"])
FLOORS = len(geom["floors"])
ORIGIN = geom["origin_lnglat"]

area = 0.5 * abs(np.dot(FP[:, 0], np.roll(FP[:, 1], -1)) - np.dot(FP[:, 1], np.roll(FP[:, 0], -1)))
INFO = {"floors": FLOORS, "below": int((prior.get("ledger") or {}).get("ugrndFlrCnt") or 0), "area_m2": area,
        "use_code": prior.get("use_code") or "", "approval": prior.get("approval_date") or "",
        "name": (prior.get("juso") or {}).get("bd_nm") or ""}

rf = OUT / "roof_items.json"
if (OUT / "roof_crop.png").exists() and rf.exists():
    ORTHO = np.array(Image.open(OUT / "roof_crop.png").convert("RGB"))
    ORTHO_AXIS = json.loads(rf.read_text(encoding="utf-8"))["axis_deg"]
    ORTHO_SRC = "4번 노트북 out/roof_crop.png"
else:
    ORTHO = np.array(Image.open(DATA / "roof_ortho.png").convert("RGB"))
    ORTHO_AXIS = json.loads((DATA / "bds.json").read_text(encoding="utf-8"))["roof"]["axis_deg"]
    ORTHO_SRC = "실제 run 의 roof_ortho.png"
ORTHO_W, ORTHO_H = ORTHO.shape[1] * 0.25, ORTHO.shape[0] * 0.25
print(f"외곽선 {len(FP)}점 · 높이 {H:.2f} m · 지상 {FLOORS}층 · 지하 {INFO['below']}층 · 건축면적 {area:.0f} m²")
print(f"지붕 영상: {ORTHO_SRC} ({ORTHO_W:.1f} × {ORTHO_H:.1f} m, 주축 {ORTHO_AXIS:.1f}° 로 세움)")

# 거리 사진 (선택). 직접 찍은 사진을 올렸으면 여기에 추가한다: [("full, facade f_0", "my.jpg"), ...]
USER_PHOTOS = []
USE_COMMONS_PHOTO = True     # 5번 노트북의 공개 라이선스 사진 (kallerna, CC BY-SA 4.0)
SHOTS = [(lab, Image.open(p).convert("RGB")) for lab, p in USER_PHOTOS]
if USE_COMMONS_PHOTO:
    SHOTS.append(("full, facade f_0 (south-west face), oblique from across Yeongdong-daero",
                  Image.open(BASE / "data" / "commons" / "Yeongdong-daero_3.jpg").convert("RGB")))
print("거리 사진", len(SHOTS), "장:", [lab for lab, _ in SHOTS])

fig, ax = plt.subplots(1, 1 + len(SHOTS), figsize=(5 + 8 * len(SHOTS), 7), gridspec_kw={"width_ratios": [1] + [3] * len(SHOTS)})
ax = np.atleast_1d(ax)
ax[0].imshow(ORTHO); ax[0].set_title("지붕 (위성)", fontsize=13); ax[0].axis("off")
for a, (lab, im) in zip(ax[1:], SHOTS):
    a.imshow(im); a.set_title(lab[:60], fontsize=12); a.axis("off")
plt.tight_layout(); plt.show()

## 3. 프롬프트와 API 문서 — 원본 그대로

모델이 받는 것은 이 텍스트와 이미지뿐이다. 마지막의 **API 문서가 모델이 아는 세계의 전부**다 — 함수 7개.
`mat` 으로 재질을 선언하고 `box`·`prism`·`cyl`·`quad`·`text` 로 형상을, `group` 으로 부품 이름을 붙인다.

원본과 한 군데 다르게 했다: 원본 프롬프트는 지붕 영상을 "north up" 이라고 소개하지만, 실제로 넘기는 영상은
건물 주축으로 **돌려 세운** 크롭이다(4번 노트북 7절). 여기서는 돌린 각도를 그대로 알려 준다.

In [ ]:
API_DOC = """\
Available functions (this is the WHOLE API — nothing else exists, no imports):

  mat(name, color, kind="plaster", roughness=None, metallic=None,
      unit=None, accent=None, emission=0.0)
      Declare a material once, then pass its name to geometry calls.
        color   "#RRGGBB"
        kind    plaster | paint | concrete | brick | tile | stone | metal_panel |
                metal | frame | glass | storefront_glass | curtain_wall | wood |
                fabric | sign | railing
        unit    [w, h] in metres — the cladding module for brick/tile/stone/metal_panel.
                This is what makes a surface read as tile vs stone. Use it.
        accent  "#RRGGBB" joint / mortar colour.
        emission 0..1, for lit signage only.

  box(x, y, z, w, d, h, m, rot=0)
      Axis-aligned box. (x, y) is the CENTRE of the footprint, z is the BOTTOM.
      w = size along local X, d = along local Y, h = height. rot = degrees CCW about Z
      around (x, y). Use rot to align with the building, not with north.

  prism(ring, z0, z1, m)
      Extrude any closed polygon. ring = [[x, y], ...] in metres, no repeated last point.
      This is how you make non-rectangular massing, curved corners (many short edges),
      setbacks, podiums, canopies.

  cyl(x, y, z, r, h, m, seg=24, axis="Z")
      Cylinder. axis "Z" vertical, or [dx, dy, dz] for a direction (pipes, railings).

  quad(p0, p1, p2, p3, m)
      One flat quad from four [x, y, z] points. For glazing planes, signs, thin panels.

  text(s, x, y, z, height, m, rot=0, depth=0.02, align="CENTER")
      3D text. Korean is fine. Only for signage you can actually READ in the photos.

  group(name)
      Start a named part from here on (e.g. group("F1_STOREFRONT")). Parts become separate
      objects in the .blend so a human can edit them. Call it often — it costs nothing.
"""

PROMPT = """\
You are building a 3D model of ONE real building, by writing a Python script.

Everything you need to judge the building's appearance is in the photographs. Look at them and
build what is actually there — the real colours, the real cladding, the real shape. Do not build a
generic building of this type.

FIXED — these come from national survey data and must not be contradicted (unless marked NOT fixed),
because this model will later be placed on a map next to other buildings:

  Coordinate frame : LOCAL_ENU. +X = east, +Y = north, +Z = up. Units are METRES.
                     Origin (0, 0, 0) = lng {lng:.6f}, lat {lat:.6f} at ground level.
  Parcel outline   : the building footprint, in this frame{ring_note}:
{ring}
                     Stay inside it. The ground floor should meet it closely; upper floors may
                     step in or cantilever slightly, as the photos show.
{height_block}
REFERENCE — register facts about this building. They tell you its SCALE, not its appearance. Where
a fact and a photograph disagree, the photograph wins.

{info}
{height_check}

EVERYTHING ELSE IS YOURS TO DECIDE FROM THE PHOTOS — the number and thickness of visible floor
bands, window shape and rhythm, whether there are windows at all, cladding material and colour,
the podium, entrances, signage, rooftop equipment, balconies, louvres, pipes.

Photographs, in order:
  1. Aerial orthophoto of the roof, {ortho_orient}, {ortho_w:.0f} m wide and {ortho_h:.0f} m tall,
     covering the parcel outline above. Use it for roof equipment and for the real outline.
{shots}

How to make it look like the building rather than like a diagram:
- Colour is the single thing people notice first. Read it off the photos, in daylight terms,
  including how glass actually reads here (green, blue, grey, mirrored).
- Give cladding a `unit` so tile reads as tile and stone as stone.
- Depth is what separates a model from a drawing. Windows sit BACK from the wall; sills, bands,
  copings, canopies and signs stick OUT. A few centimetres is enough.
- Build the sides and back too. They are usually plainer, with service items — but they exist,
  and the model will be seen from every angle.
- Repetition belongs in a `for` loop. A 9-storey facade is a loop, not 400 pasted lines.

Output ONLY the Python script. No prose, no markdown fence, no imports. Assume the API below is
already in scope. The script runs sandboxed: no imports, no `while`, no lambda, and the only
attributes you may use are ordinary list/str/dict methods (.append, .join, .format ...). Plain
variables, `for`, `if`, arithmetic, f-strings and helper `def`s are all fine.

{api}"""

USE_NAMES = {"01": "단독주택", "02": "공동주택", "03": "제1종 근린생활시설", "04": "제2종 근린생활시설",
             "05": "문화·집회시설", "07": "판매시설", "09": "의료시설", "10": "교육연구시설",
             "14": "업무시설", "15": "숙박시설", "17": "공장", "18": "창고시설"}

def info_block(d):
    rows = []
    if d["name"]:
        rows.append(f"  Building name  : {d['name']}")
    rows.append(f"  Floors         : {d['floors']} above ground" + (f", {d['below']} below ground" if d["below"] else ""))
    if d["use_code"]:
        nm = USE_NAMES.get(str(d["use_code"])[:2], "")
        rows.append(f"  Registered use : {d['use_code']}{f' ({nm})' if nm else ''}"
                    " — what happens inside; it does NOT dictate how the outside looks")
    if d["approval"]:
        rows.append(f"  Approved       : {d['approval']} — tells you the era of construction")
    rows.append(f"  Footprint area : {d['area_m2']:.0f} m2")
    return "\n".join(rows)

def make_prompt(feedback=""):
    ring = "\n".join(f"                       [{x:8.2f}, {y:8.2f}]," for x, y in FP)
    orient = ("north up" if abs(ORTHO_AXIS) < 0.75 else
              f"rotated so the building's main axis is horizontal — image up points {ORTHO_AXIS:.0f} degrees "
              f"counter-clockwise from north (bearing {(360 - ORTHO_AXIS) % 360:.0f})")
    p = PROMPT.format(
        lng=ORIGIN[0], lat=ORIGIN[1], ring=ring, ring_note="",
        height_block=f"  Height           : about {H:.1f} m to the top of the main roof.\n",
        height_check=(f"  A useful check: {H:.1f} m over {FLOORS} floors is about {H / FLOORS:.1f} m per floor. Count the\n"
                      "  floor lines you can actually see in the photos — if you see a different number, say so with the\n"
                      "  geometry you build, but keep the total height."),
        info=info_block(INFO), ortho_orient=orient, ortho_w=ORTHO_W, ortho_h=ORTHO_H,
        shots="\n".join(f"  {i + 2}. street-level photo — {lab}" for i, (lab, _) in enumerate(SHOTS)), api=API_DOC)
    if feedback:
        p += ("\n\nYour previous script was rejected. Fix exactly these problems and output "
              "the whole corrected script again:\n" + feedback)
    return p

print(make_prompt()[:2600], "\n...\n(이하 API 문서)")

## 4. 정적 검사기 — 실행하기 **전에** 코드를 읽는다

남이 쓴 코드를 실행하는 것은 위험하다. 모델이 악의가 없어도, 프롬프트에 끼어든 글 한 줄이 `import os` 를 쓰게 만들 수 있다.
그래서 실행 전에 파이썬 **구문 트리(AST)** 를 훑어 허용 목록 밖의 것을 전부 거부한다 (`check_script` 이식).

* 임포트 금지, `while` · `lambda` · `global` 금지 — 무한 반복과 바깥 이름 덮어쓰기를 막는다.
* 호출은 **허용된 이름**(API 7개 + 기본 함수 + 수학)과 스크립트가 직접 `def` 한 함수만.
* 속성 접근(`.xxx`)은 리스트·문자열·딕셔너리 메서드만. **던더(`__xxx__`)는 어떤 경우에도 금지** —
  `().__class__.__bases__[0].__subclasses__()` 로 샌드박스를 빠져나가는 고전적 수법을 막는다.

In [ ]:
ALLOWED_ATTRS = {
    "append", "extend", "insert", "pop", "copy", "sort", "reverse", "index", "count",
    "add", "update", "keys", "values", "items", "get",
    "join", "format", "strip", "lstrip", "rstrip", "lower", "upper", "split", "replace",
}
ALLOWED_CALLS = {
    "mat", "box", "prism", "cyl", "quad", "text", "group",
    "range", "len", "min", "max", "abs", "round", "int", "float", "str", "enumerate", "zip",
    "sum", "sorted", "reversed", "list", "tuple", "dict", "set", "bool",
    "sin", "cos", "tan", "atan2", "hypot", "sqrt", "radians", "degrees", "pi", "floor", "ceil",
}

def check_script(code):
    """실행 전 정적 검사. 반환: 문제 목록 (비어 있으면 통과)."""
    problems = []
    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        return [f"문법 오류 {e.lineno}행: {e.msg}"]
    own = {n.name for n in ast.walk(tree) if isinstance(n, ast.FunctionDef)}   # 스크립트가 정의한 함수
    allowed_calls = ALLOWED_CALLS | own
    for node in ast.walk(tree):
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            problems.append(f"{node.lineno}행: 임포트 금지")
        elif isinstance(node, ast.Attribute):
            if node.attr.startswith("__") or node.attr not in ALLOWED_ATTRS:
                problems.append(f"{node.lineno}행: 허용되지 않은 속성 .{node.attr}")
        elif isinstance(node, ast.Name) and node.id.startswith("__"):
            problems.append(f"{node.lineno}행: 예약 이름 사용 ({node.id})")
        elif isinstance(node, ast.Call) and isinstance(node.func, ast.Name):
            if node.func.id not in allowed_calls:
                problems.append(f"{node.lineno}행: 허용되지 않은 호출 {node.func.id}()")
        elif isinstance(node, (ast.Global, ast.Nonlocal)):
            problems.append(f"{node.lineno}행: global/nonlocal 금지")
        elif isinstance(node, (ast.While, ast.Lambda)):
            problems.append(f"{node.lineno}행: while/lambda 금지")
    return problems[:12]

### 검사기를 공격해 본다

샌드박스 탈출에 실제로 쓰이는 수법들이다. 전부 **실행되기 전에** 걸러져야 한다.

In [ ]:
ATTACKS = {
    "import os":            "import os\nos.system('echo hacked')",
    "__import__":           "__import__('os').system('echo hacked')",
    "().__class__ 탈출":     "subs = ().__class__.__bases__[0].__subclasses__()",
    "while True":           "while True:\n    box(0, 0, 0, 1, 1, 1, 'm')",
    "open() 파일 읽기":      "data = open('/etc/passwd').read()",
    "eval":                 "eval('1 + 1')",
    "getattr 우회":          "g = getattr(box, '__globals__')",
    "함수 속성 훔쳐보기":     "c = box.__code__",
    "lambda":               "f = lambda: 0",
    "global 로 API 덮어쓰기": "def f():\n    global box\n    box = None",
    "평범한 스크립트":        "mat('w', '#cccccc')\nfor i in range(3):\n    box(0, 0, i * 3.0, 10, 8, 3, 'w')",
}
for name, src in ATTACKS.items():
    p = check_script(src)
    print(f"{'통과' if not p else '거부':4s}  {name:22s} {'; '.join(p)}")

### 왜 검사기와 빈 `__builtins__` 를 **둘 다** 쓰나

샌드박스는 `exec(code, {"__builtins__": {}, ...})` 로 내장 함수를 전부 뺀 이름공간에서 돈다.
그러면 `open`, `__import__` 는 이름 자체가 없다. 그런데 그것만으로는 부족하다 — **객체의 속성을 타고 올라가면**
내장 함수 없이도 파이썬 안의 모든 클래스에 손이 닿는다. 검사기가 던더 속성을 막는 이유다.

그리고 검사기가 통과시키는 것도 위험할 수 있다: `range(10**9)` 는 합법이지만 노트북을 멈춘다.
그래서 세 번째 줄로, 샌드박스 안의 `range` 는 길이 10만을 넘으면 거부하고 빌더는 120초를 넘으면 멈춘다 (이 노트북에서 추가).

In [ ]:
for src in ["open('/etc/passwd')", "__import__('os')"]:
    try:
        exec(src, {"__builtins__": {}})
    except NameError as e:
        print(f"빈 __builtins__ : {src:22s} → 막힘 ({e})")

env = {"__builtins__": {}}
exec("subs = ().__class__.__bases__[0].__subclasses__()", env)       # 검사기가 없다면 실행된다
risky = sorted({c.__name__ for c in env["subs"] if c.__name__ in ("Popen", "FileIO", "BuiltinImporter", "FileLoader", "SourceFileLoader")})
print(f"빈 __builtins__ 만으로는: 클래스 {len(env['subs'])}개에 손이 닿는다 — 그중 {risky}")
print("→ 그래서 검사기가 던더 속성을 실행 전에 막는다:", check_script("subs = ().__class__.__bases__[0].__subclasses__()")[:1])

## 5. 빌더 — 7개 함수를 trimesh 로 구현한다

원본 `blender/build_freeform.py` 의 `Builder` 를 옮겼다. 스크립트는 이 7개 함수와 수학 함수만 볼 수 있다.

* `box` — 중심 (x, y), 바닥 z, `rot` 은 z 축 반시계 회전(도). Blender 판과 꼭짓점 순서까지 같다.
* `prism` — 옆면은 사각형, 위·아래 뚜껑은 **귀 자르기(earcut)** 삼각분할. 원본은 중심점 부채꼴이라 오목한 평면에서 뚜껑이 밖으로 삐져나온다.
* `cyl` — 축 방향 원기둥 (배관·난간).
* `text` — 글꼴 윤곽(matplotlib `TextPath`)을 밀어 올린 진짜 3D 글자. 한글 글꼴이 없으면 같은 크기 판으로 대신한다.
* 재질의 `unit`·`accent`(타일 줄눈)는 Blender 절차적 재질이 그린다. trimesh 판은 색·거칠기·금속감만 쓴다.
* 부재 상한 30,000 — 넘으면 예외 대신 **거기까지 지은 것**을 내보낸다 (6~7분 걸린 생성을 통째로 버리지 않도록).

In [ ]:
MAX_PARTS, MAX_TEXT = 30000, 60
KIND_PBR = {"glass": (0.06, 0.4), "storefront_glass": (0.05, 0.2), "curtain_wall": (0.1, 0.4), "metal": (0.3, 0.85),
            "metal_panel": (0.35, 0.6), "frame": (0.35, 0.65), "railing": (0.4, 0.7), "sign": (0.45, 0.0)}

from matplotlib.textpath import TextPath
from matplotlib.font_manager import FontProperties
TEXT_FONT = FontProperties(fname=FONT_PATH) if FONT_PATH and os.path.exists(FONT_PATH) else FontProperties()

def text_polygons(s, size):
    """글자 윤곽 → shapely 다각형 (구멍은 겹침 홀짝으로)."""
    tp = TextPath((0, 0), s, size=size, prop=TEXT_FONT)
    geom_ = None
    for ring in tp.to_polygons():
        if len(ring) < 3:
            continue
        p = Polygon(ring).buffer(0)
        if p.is_empty:
            continue
        geom_ = p if geom_ is None else geom_.symmetric_difference(p)
    return geom_

class Builder:
    """생성 스크립트에 노출되는 전부."""
    def __init__(self, cap=MAX_PARTS, time_limit=120):
        self.ps, self.defs, self.key = PartSet(), {}, {}
        self.cur, self.n, self.texts, self.over = "BODY", 0, 0, False
        self.warn, self.cap, self.t0, self.limit = [], cap, time.time(), time_limit
        self.calls, self.text_log = Counter(), []

    # ---- 내부
    def _m(self, name):
        name = str(name)
        if name in self.key:
            return self.key[name]
        msg = f"정의되지 않은 재질 '{name}' → 기본 미장"
        if msg not in self.warn:
            self.warn.append(msg)
        self.ps.material("_undefined", "#C9C4BA", 0.85, 0.0)
        return "_undefined"

    def _budget(self, kind):
        self.calls[kind] += 1
        if time.time() - self.t0 > self.limit:
            raise TimeoutError(f"빌드 {self.limit}초 초과")
        self.n += 1
        if self.n > self.cap:
            if not self.over:
                self.over = True
                self.warn.append(f"부재 {self.cap}개 상한에 도달해 이후 호출은 그리지 않았습니다")
            return False
        return True

    def _add(self, m, V, F):
        self.ps.add(self.cur, self._m(m), V, F)

    # ---- 공개 API
    def group(self, name):
        self.calls["group"] += 1
        self.cur = "".join(c if (c.isalnum() or c in "_-") else "_" for c in str(name))[:40] or "BODY"

    def mat(self, name, color, kind="plaster", roughness=None, metallic=None, unit=None, accent=None, emission=0.0):
        self.calls["mat"] += 1
        name = str(name)
        rough, metal = KIND_PBR.get(str(kind), (0.8, 0.0))
        k = f"{name}~{len(self.defs)}"          # 같은 이름을 다시 선언하면 그 뒤 형상부터 새 재질
        self.ps.material(k, str(color), rough if roughness is None else float(roughness),
                         metal if metallic is None else float(metallic))
        self.defs[k] = {"name": name, "kind": str(kind), "color": str(color), "unit": unit, "accent": accent,
                        "emission": float(emission or 0)}
        self.key[name] = k

    def box(self, x, y, z, w, d, h, m, rot=0):
        if not self._budget("box"):
            return
        a = math.radians(float(rot))
        c, s = math.cos(a), math.sin(a)
        u = np.array([c * float(w), s * float(w), 0.0])
        v = np.array([-s * float(d), c * float(d), 0.0])
        o = np.array([float(x), float(y), float(z)]) - u / 2 - v / 2
        self._add(m, *box_vf(o, u, v, [0, 0, float(h)]))

    def prism(self, ring, z0, z1, m):
        if not self._budget("prism"):
            return
        pts = np.array([[float(p[0]), float(p[1])] for p in ring])
        if len(pts) < 3:
            return
        z0, z1 = float(z0), float(z1)
        n = len(pts)
        V = np.vstack([np.column_stack([pts, np.full(n, z0)]), np.column_stack([pts, np.full(n, z1)])])
        F = []
        for i in range(n):                      # 옆면
            j = (i + 1) % n
            F += [(i, j, n + j), (i, n + j, n + i)]
        try:                                     # 뚜껑: earcut (오목해도 정확)
            v2, f2 = trimesh.creation.triangulate_polygon(Polygon(pts))
            if len(v2) != n:
                raise ValueError("꼭짓점 수가 바뀜")
            idx = [int(np.argmin(np.hypot(*(pts - q).T))) for q in v2]
            for a_, b_, c_ in f2:
                F += [(n + idx[a_], n + idx[b_], n + idx[c_]), (idx[c_], idx[b_], idx[a_])]
        except Exception:                        # 꼬인 다각형이면 원본처럼 중심 부채꼴
            V = np.vstack([V, [[*pts.mean(0), z0], [*pts.mean(0), z1]]])
            for i in range(n):
                j = (i + 1) % n
                F += [(2 * n + 1, n + i, n + j), (2 * n, j, i)]
        self._add(m, V, np.array(F))

    def cyl(self, x, y, z, r, h, m, seg=24, axis="Z"):
        if not self._budget("cyl"):
            return
        seg = max(6, min(64, int(seg)))
        c = np.array([float(x), float(y), float(z)])
        up = np.array([0, 0, 1.0]) if isinstance(axis, str) else np.array([float(a) for a in axis])
        up = up / (np.linalg.norm(up) or 1.0)
        ref = np.array([1.0, 0, 0]) if abs(up[2]) > 0.9 else np.array([0, 0, 1.0])
        e1 = np.cross(up, ref); e1 /= np.linalg.norm(e1)
        e2 = np.cross(up, e1)
        ang = 2 * np.pi * np.arange(seg) / seg
        ring_b = c + float(r) * (np.outer(np.cos(ang), e1) + np.outer(np.sin(ang), e2))
        V = np.vstack([ring_b, ring_b + up * float(h)])
        F = []
        for i in range(seg):
            j = (i + 1) % seg
            F += [(i, j, seg + j), (i, seg + j, seg + i)]
        for i in range(1, seg - 1):
            F += [(0, i + 1, i), (seg, seg + i, seg + i + 1)]
        self._add(m, V, np.array(F))

    def quad(self, p0, p1, p2, p3, m):
        if not self._budget("quad"):
            return
        V = np.array([[float(c) for c in p] for p in (p0, p1, p2, p3)])
        self._add(m, V, np.array([(0, 1, 2), (0, 2, 3)]))

    def text(self, s, x, y, z, height, m, rot=0, depth=0.02, align="CENTER"):
        self.calls["text"] += 1
        if self.texts >= MAX_TEXT or not str(s).strip():
            return
        self.texts += 1
        s, hgt, dep = str(s)[:60], max(0.05, float(height)), max(0.005, float(depth))
        self.text_log.append(s)
        try:
            g = text_polygons(s, hgt)
            minx, miny, maxx, maxy = g.bounds
            parts = list(getattr(g, "geoms", [g]))
            Vs, Fs, off = [], [], 0
            for p in parts:
                mesh = trimesh.creation.extrude_polygon(p, 2 * dep)
                Vs.append(mesh.vertices); Fs.append(mesh.faces + off); off += len(mesh.vertices)
            V, F = np.vstack(Vs), np.vstack(Fs)
        except Exception:                        # 글꼴이 없으면 같은 크기 판
            minx, miny, maxx, maxy = 0, 0, 0.6 * hgt * len(s), hgt
            V, F = box_vf([0, 0, 0], [maxx, 0, 0], [0, maxy, 0], [0, 0, 2 * dep])
        dx = {"LEFT": -minx, "RIGHT": -maxx}.get(align, -(minx + maxx) / 2)
        lx, ly, lz = V[:, 0] + dx, V[:, 1] - (miny + maxy) / 2, V[:, 2] - dep
        # Blender 와 같은 자세: x 축으로 90° 세우고 (글자 위 = +Z, 두께 = −Y), z 축으로 rot
        a = math.radians(float(rot))
        wx, wy, wz = lx, -lz, ly
        V = np.column_stack([float(x) + wx * math.cos(a) - wy * math.sin(a),
                             float(y) + wx * math.sin(a) + wy * math.cos(a), float(z) + wz])
        self._add(m, V, F)

def run_script(code, cap=MAX_PARTS, time_limit=120):
    """생성 스크립트를 제한된 이름공간에서 실행한다."""
    b = Builder(cap, time_limit)
    def safe_range(*a):
        r = range(*a)
        if len(r) > 100_000:
            raise ValueError(f"range 길이 {len(r):,} — 10만을 넘는 반복은 거부")
        return r
    env = {
        "mat": b.mat, "box": b.box, "prism": b.prism, "cyl": b.cyl, "quad": b.quad, "text": b.text, "group": b.group,
        "sin": math.sin, "cos": math.cos, "tan": math.tan, "atan2": math.atan2, "hypot": math.hypot,
        "sqrt": math.sqrt, "radians": math.radians, "degrees": math.degrees, "pi": math.pi,
        "floor": math.floor, "ceil": math.ceil,
        "range": safe_range, "len": len, "min": min, "max": max, "abs": abs, "round": round,
        "int": int, "float": float, "str": str, "enumerate": enumerate, "zip": zip, "sum": sum,
        "sorted": sorted, "reversed": reversed, "list": list, "tuple": tuple, "dict": dict, "set": set, "bool": bool,
        "__builtins__": {},                      # 내장 전부 차단
    }
    exec(compile(code, "<generated>", "exec"), env, env)
    return b

# 합법이지만 위험한 코드: 검사기는 통과시키고, 샌드박스의 range 가 막는다
try:
    print("검사기:", check_script("for i in range(10**9):\n    pass") or "통과 (합법적인 코드다)")
    run_script("for i in range(10**9):\n    pass")
except ValueError as e:
    print("샌드박스:", e)

## 6. 모델에게 스크립트를 받는다

모델 이름은 변수다. 실제 파이프라인 설정(`configs/default.yaml` 의 `freeform.model`)은 `gpt-6-astra` 이고,
추론 강도 high, 출력 상한 32,000 토큰이다. 30층 타워면 6~7분 걸린다.
**가진 비전 모델이 다르면 이름만 바꾸면 된다** (예: 파이프라인의 판독 모델 `gpt-5.5`). 모델마다 스크립트가 얼마나 다른지가 이 노트북의 관전 포인트다.

검사에 걸리면 문제 목록을 그대로 돌려주고 **한 번 더** 쓰게 한다 (원본 `retries: 1`).
호출이 실패하거나 두 번 다 걸리면, 실제 run 의 스크립트로 넘어간다.

In [ ]:
def to_data_url(img, max_side=1024, fmt="PNG"):
    """numpy 배열·PIL 이미지 → data URL (긴 변 max_side 이하로 줄여서)."""
    im = Image.fromarray(img) if isinstance(img, np.ndarray) else img.convert("RGB")
    s = max_side / max(im.size)
    if s < 1:
        im = im.resize((round(im.width * s), round(im.height * s)), Image.LANCZOS)
    buf = io.BytesIO()
    im.save(buf, fmt, **({"quality": 90} if fmt == "JPEG" else {}))
    return f"data:image/{fmt.lower()};base64," + base64.b64encode(buf.getvalue()).decode()

ASTRA_MODEL = "gpt-6-astra"
REASONING = "high"            # 추론 모델이 아니면 None
MAX_OUTPUT_TOKENS = 32000
TIMEOUT_S = 900

def strip_fence(code):
    code = code.strip()
    if code.startswith("```"):
        code = code.split("\n", 1)[-1].rsplit("```", 1)[0]
    return code.strip()

SCRIPT, GEN_META, problems = None, {}, []
if OPENAI_KEY:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_KEY, timeout=TIMEOUT_S, max_retries=0)
    for rnd in range(2):
        content = [{"type": "input_text", "text": make_prompt("\n".join(problems))},
                   {"type": "input_image", "image_url": to_data_url(ORTHO, 1024)}]
        content += [{"type": "input_image", "image_url": to_data_url(im, 1400, "JPEG")} for _, im in SHOTS]
        kwargs = dict(model=ASTRA_MODEL, input=[{"role": "user", "content": content}], max_output_tokens=MAX_OUTPUT_TOKENS)
        if REASONING:
            kwargs["reasoning"] = {"effort": REASONING}
        try:
            t0 = time.time()
            resp = client.responses.create(**kwargs)
            code = strip_fence(resp.output_text or "")
        except Exception as e:
            print(f"{ASTRA_MODEL} 호출 실패 →", type(e).__name__, str(e)[:400])
            break
        GEN_META = {"source": f"openai:{ASTRA_MODEL}", "round": rnd + 1, "latency_s": round(time.time() - t0, 1),
                    "input_tokens": resp.usage.input_tokens, "output_tokens": resp.usage.output_tokens}
        problems = check_script(code)
        print(f"{rnd + 1}회차: {GEN_META['latency_s']}초 · {code.count(chr(10)) + 1}행 · 토큰 입력 {resp.usage.input_tokens:,} / 출력 {resp.usage.output_tokens:,}"
              + (f" · 반려 {len(problems)}건: {problems[:3]}" if problems else " · 검사 통과"))
        if not problems:
            SCRIPT = code
            break
if SCRIPT is None:
    SCRIPT = (DATA / "astra_build.py").read_text(encoding="utf-8")
    GEN_META = {"source": "staged:astra_build.py (실제 run, gpt-6-astra)"}
    print("실제 run 의 astra 스크립트로 진행 — 같은 검사기와 샌드박스를 통과해야 한다")
_ = (OUT / "astra_build.py").write_text(SCRIPT, encoding="utf-8")

## 7. 스크립트 읽기 — 실행 전에 무엇을 하려는지 본다

In [ ]:
tree = ast.parse(SCRIPT)
api_calls = Counter(n.func.id for n in ast.walk(tree) if isinstance(n, ast.Call) and isinstance(n.func, ast.Name)
                    and n.func.id in {"mat", "box", "prism", "cyl", "quad", "text", "group"})
defs = [n.name for n in ast.walk(tree) if isinstance(n, ast.FunctionDef)]
loops = sum(isinstance(n, ast.For) for n in ast.walk(tree))
lines = SCRIPT.count("\n") + 1
problems = check_script(SCRIPT)
groups = re.findall(r'group\("([^"]+)"\)', SCRIPT)

print(f"출처      {GEN_META['source']}")
print(f"분량      {lines}행 · {len(SCRIPT):,}자 · for 반복 {loops}개 · 직접 정의한 함수 {len(defs)}개 {defs}")
print(f"API 호출  (코드에 적힌 횟수) " + ", ".join(f"{k} {v}" for k, v in api_calls.most_common()))
print(f"검사기    {'통과' if not problems else problems}")
print(f"부품 이름 ({len(groups)}) " + ", ".join(groups))
print("\n── 처음 30행 ──")
print("\n".join(SCRIPT.splitlines()[:30]))

## 8. 샌드박스에서 짓는다

In [ ]:
if problems:
    raise SystemExit("검사를 통과하지 못한 스크립트는 실행하지 않는다")
t0 = time.time()
try:
    B = run_script(SCRIPT)
except Exception as e:
    raise SystemExit(f"생성 스크립트 실행 오류 — {type(e).__name__}: {e}")
st = B.ps.stats()
lo, hi = B.ps.bounds()
T, _ = B.ps.triangles()
verts = T.reshape(-1, 3)
parcel = Polygon(FP).buffer(1.5)
inside = np.mean([parcel.contains(Point(p)) for p in verts[:: max(1, len(verts) // 4000), :2]])

print(f"실행 {time.time() - t0:.1f}초 · 부재 {B.n:,}개 ({', '.join(f'{k} {v}' for k, v in B.calls.most_common())})")
print(f"부품 {st['groups']}개 · 재질 {len(B.defs)}종 · 삼각형 {st['triangles']:,}개 · 3D 글자 {B.texts}개 {B.text_log}")
print(f"지은 크기 x {lo[0]:.1f}~{hi[0]:.1f}, y {lo[1]:.1f}~{hi[1]:.1f} m · 높이 {hi[2] - min(0, lo[2]):.2f} m (대장 {H:.2f} m, 차이 {hi[2] - H:+.2f} m)")
print(f"대지 안(+1.5 m)에 든 꼭짓점 비율 {inside * 100:.1f} %")
for w in B.warn[:6]:
    print("경고:", w)

In [ ]:
GLB = B.ps.glb()
(OUT / "astra.glb").write_bytes(GLB)
img = render(B.ps, size=(1280, 720), azim=200, elev=10)
Image.fromarray(img).save(OUT / "astra.png")
plt.figure(figsize=(16, 9)); plt.imshow(img); plt.axis("off")
plt.title(f"모델이 쓴 스크립트로 지은 건물 — {GEN_META['source']}", fontsize=15); plt.show()
show_glb(GLB, note="마우스로 돌려 볼 수 있다. 인터넷이 없으면 빈 칸 — 위 정적 렌더를 볼 것.")

## 9. 같은 건물, 두 방식

5번 노트북을 돌렸다면 나란히 본다. 왼쪽은 **스키마가 허락한 만큼**, 오른쪽은 **모델이 사진에서 본 만큼**이다.
어느 쪽이 "맞는가"는 사진과 비교해야 한다 — 그리고 오른쪽은 **같은 입력에도 매번 다른 스크립트**가 나온다는 것을 잊지 말 것.

In [ ]:
panels = [("v0.3 BDS → 결정론적 모델러", OUT / "ipark_bds.png"), ("v0.4 모델이 쓴 스크립트", OUT / "astra.png")]
panels = [(t, p) for t, p in panels if p.exists()]
fig, ax = plt.subplots(1, len(panels) + 1, figsize=(8 * (len(panels) + 1), 5.5))
ph = BASE / "data" / "commons" / "Yeongdong-daero_3.jpg"
ax[0].imshow(Image.open(ph)); ax[0].set_title("실제 (kallerna, CC BY-SA 4.0)", fontsize=14)
for a, (t, p) in zip(ax[1:], panels):
    im = np.array(Image.open(p))
    a.imshow(im[:, im.shape[1] // 6: -im.shape[1] // 6]); a.set_title(t, fontsize=14)
for a in ax:
    a.axis("off")
plt.tight_layout(); plt.show()

## 10. 저장

In [ ]:
stats = {"source": GEN_META, "lines": lines, "chars": len(SCRIPT), "loops": loops, "defs": defs,
         "api_calls_static": dict(api_calls), "api_calls_runtime": dict(B.calls), "parts": B.n,
         "groups": st["groups"], "materials": len(B.defs), "triangles": st["triangles"], "texts": B.text_log,
         "built_height_m": round(float(hi[2]), 2), "register_height_m": H, "inside_parcel_ratio": round(float(inside), 3),
         "warnings": B.warn, "glb_sha256_16": sha(GLB)}
(OUT / "astra_stats.json").write_text(json.dumps(stats, ensure_ascii=False, indent=1), encoding="utf-8")

print("── 요약 ─────────────────────────────")
print(f"스크립트   {GEN_META['source']} · {lines}행 · 검사 통과")
print(f"실행       부재 {B.n:,} · 부품 {st['groups']} · 재질 {len(B.defs)} · 삼각형 {st['triangles']:,}")
print(f"높이       지은 {hi[2]:.2f} m / 대장 {H:.2f} m")
print(f"검사기     공격 {sum(1 for s in ATTACKS.values() if check_script(s))}/{len(ATTACKS) - 1} 거부, 평범한 스크립트 통과")
for f in ["astra_build.py", "astra.glb", "astra.png", "astra_stats.json"]:
    print("  out/" + f, f"{(OUT / f).stat().st_size / 1024:.0f} KB")

## 해 볼 것

1. `ASTRA_MODEL` 을 가진 다른 비전 모델로 바꿔 돌려 보자. 줄 수·부품 수·높이가 어떻게 다른가? 같은 모델로 두 번 돌리면?
2. `USE_COMMONS_PHOTO = False` 로 **지붕 영상만** 주면 모델은 입면을 어떻게 짓는가? 사진이 무엇을 바꾸는지 보자.
3. 검사기를 뚫어 보자. `ATTACKS` 에 새 수법을 넣고 `check_script` 가 거부하는지 확인. 통과했다면 샌드박스는 막는가?
4. 5번(BDS)과 6번(스크립트)의 결과를 사진과 비교해 점수를 매긴다면 무엇을 재야 할까? (실루엣, 색, 창 격자, 높이…)
5. "모델이 쓴 코드를 실행한다"는 설계의 장단점을 정리해 보자. 도시 전체(건물 수만 동)로 늘리면 무엇이 문제가 되나?

### 참고
* addr2blend `src/addr2blend/bds/freeform.py` (프롬프트·API·검사기), `blender/build_freeform.py` (샌드박스)
* Python `ast` 모듈 문서 — 구문 트리 노드 종류
* 예제 사진: *Yeongdong-daero 3* — kallerna, Wikimedia Commons, CC BY-SA 4.0